# 07 · JAX optical-model tools

Every notebook so far has leaned on one thing: a model that says *where light of a given wavelength lands on the detector*. This notebook opens it up. The optical model is written in **JAX**, so it isn't just a lookup — it is a small, **differentiable**, **vectorizable** set of functions. That is what lets us get the dispersion by autodiff (notebook 04), warp galaxy stamps through the dispersion Jacobian (notebook 05's morphology effects), and run the whole thing on a GPU (notebook 06).

We work entirely in the optical model here — no PSFs, no flux — so everything runs in seconds.

## 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from roman_disperser import paths
from roman_disperser.elements import GRISM
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj

SCA = 5
element = GRISM
model = RomanOpticalModel(config_file=str(paths.optical_model_path(element=element)))
payload = omj.make_sca_payload(model, sca=SCA, order="1")
print("payload keys:", list(payload.keys()), "| JAX backend:", jax.default_backend())

## 1 · Three coordinate frames

The model moves between three frames, and the disperser/extractor hop along this chain constantly:

- **SCA pixels** — detector pixels (1-indexed FITS), what you actually read out;
- **FPA degrees** — angle on the sky-side focal plane, shared across all 18 detectors;
- **MPA millimetres** — physical position on the mosaic plate, where the grism dispersion polynomial is defined.

`optical_model_jax` exposes the pairwise transforms (`sca_to_fpa`, `fpa_to_mpa`, `mpa_to_sca`, and their inverses), each taking the payload first. Because they are proper inverses, a round trip returns the input exactly.

In [ ]:
x, y = jnp.array([2000.0]), jnp.array([1500.0])
xf, yf = omj.sca_to_fpa(payload, x, y)
xm, ym = omj.fpa_to_mpa(payload, xf, yf)
xb, yb = omj.mpa_to_sca(payload, xm, ym)            # all the way back to SCA pixels

print(f"SCA  ({float(x[0]):.1f}, {float(y[0]):.1f})  pix")
print(f"FPA  ({float(xf[0]):.5f}, {float(yf[0]):.5f})  deg")
print(f"MPA  ({float(xm[0]):.3f}, {float(ym[0]):.3f})  mm")
print(f"round trip back to SCA: residual = "
      f"({float(abs(xb[0]-x[0])):.2e}, {float(abs(yb[0]-y[0])):.2e}) pix")

## 2 · Sky → detector

A pointing is (RA, Dec, position angle). `get_fpa_pos(ra, dec, pointing_ra, pointing_dec, pointing_pa)` rotates sky coordinates into the focal plane (this is where the **roll** enters — the rigorous version of notebook 03, rotating about **WFICEN**, the WFI field centre), and `fpa_to_sca` then tells you which detector pixel — if any — a source falls on.

Since v0.12.0 this is an exact **gnomonic (TAN) projection** about the pointing centre, valid anywhere on the sky (earlier releases used a flat-sky approximation that was only correct near Dec = 0 — we quantify the difference below). Two design choices are worth knowing because they *will* surface as errors if you fight them:

- `get_fpa_pos` is deliberately split into a **host float64** half (`sky_to_tangent_offsets` — sky differencing, where float32 would cost up to several pixels) and a **JAX float32** half (`get_fpa_pos_from_offsets` — the roll rotation, on offsets small enough that float32 is exact to ~10⁻³ px). Feed it **float64 NumPy** arrays; float32 or JAX-array input raises a `TypeError` rather than silently losing precision.
- Consequently `get_fpa_pos` itself is **not jittable/vmappable** — jit `get_fpa_pos_from_offsets` instead. It runs once per pointing, so it is never on a hot path.

To see the geometry, we trace each SCA's four corners into the focal plane and draw the 18-detector footprint, with **WFICEN** at the centre.

In [ ]:
PRA, PDEC, PPA = 10.0, 0.0, 0.0
corners = [(1, 1), (4088, 1), (4088, 4088), (1, 4088), (1, 1)]

fig, ax = plt.subplots(figsize=(6.5, 6))
for det in range(1, 19):
    p = omj.make_sca_payload(model, sca=det, order="1")
    cx = jnp.array([c[0] for c in corners], float)
    cy = jnp.array([c[1] for c in corners], float)
    fx, fy = omj.sca_to_fpa(p, cx, cy)
    fx, fy = np.asarray(fx), np.asarray(fy)
    ax.plot(fx, fy, color="C0", lw=0.8)
    ax.text(fx[:-1].mean(), fy[:-1].mean(), str(det), ha="center", va="center", fontsize=8)
ax.plot(0, 0, "r+", ms=12, label="WFICEN (field centre)")
ax.set(xlabel="FPA x [deg]", ylabel="FPA y [deg]", title="Roman WFI focal plane — 18 SCAs")
ax.set_aspect("equal"); ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()

**How much did the projection matter?** Before v0.12.0 the sky→tangent-plane step was the flat-sky approximation $(\Delta\alpha\cos\delta,\ \Delta\delta)$, whose error is third order in the field offset at the equator but picks up a *second-order* North term $\Delta\alpha^2 \sin(2\delta_0)/4$ off it. We can reproduce that error directly — apply both projections to the same WFI-sized field of sources and difference them, converting degrees to pixels at the WFI scale of 0.11″/pix. At Dec 0 the two agree to a fraction of a pixel; at Dec 60 the flat-sky placement is off by tens of pixels at the field edge. Any product simulated off the equator with a pre-v0.12 disperser carries exactly this error.

In [ ]:
# exact gnomonic (v0.12.0+) vs the pre-v0.12 flat-sky approximation, same field
rng = np.random.default_rng(42)
n = 5000
PRA2, PPA2 = 150.0, 0.0
xi_src = rng.uniform(-0.4, 0.4, n)     # WFI-sized field, tangent-plane degrees
eta_src = rng.uniform(-0.4, 0.4, n)

for PDEC2 in (0.0, 30.0, 60.0):
    dec = PDEC2 + eta_src
    ra = PRA2 + xi_src / np.cos(np.deg2rad(dec))
    xi, eta = omj.sky_to_tangent_offsets(ra, dec, PRA2, PDEC2)     # exact (float64, host)
    xi_flat = (ra - PRA2) * np.cos(np.deg2rad(dec))                # flat sky: Δα·cosδ
    eta_flat = dec - PDEC2                                         #           Δδ
    dpix = np.hypot(np.asarray(xi) - xi_flat, np.asarray(eta) - eta_flat) * 3600 / 0.11
    print(f"Dec {PDEC2:4.0f}°:  flat-sky placement error  median {np.median(dpix):6.2f} px,  max {dpix.max():6.1f} px")

## 3 · The spectral trace

`trace_beam(payload, xfpa, yfpa, wavelength)` is the heart of dispersion: for a source at a focal-plane position it returns where each **wavelength** (in microns) lands. Chaining `sca_to_fpa → trace_beam → mpa_to_sca` gives the trace in detector pixels — exactly what notebook 04 inverted to extract a spectrum.

In [ ]:
wl = jnp.linspace(element.lam_min, element.lam_max, 60)
xf0, yf0 = omj.sca_to_fpa(payload, jnp.array([2000.0]), jnp.array([2000.0]))
xm, ym = omj.trace_beam(payload, jnp.broadcast_to(xf0, wl.shape),
                        jnp.broadcast_to(yf0, wl.shape), wl)
tx, ty = omj.mpa_to_sca(payload, xm, ym)
tx, ty = np.asarray(tx), np.asarray(ty)

fig, (a0, a1) = plt.subplots(1, 2, figsize=(11, 3.6))
a0.plot(np.asarray(wl), ty, color="C1")
a0.set(xlabel="wavelength [µm]", ylabel="trace y [pix]", title="dispersion runs along y")
a1.plot(tx, ty, color="C1")
a1.plot(2000, 2000, "k+", ms=9, label="source")
a1.set(xlabel="trace x [pix]", ylabel="trace y [pix]", title="trace on the detector"); a1.legend()
fig.tight_layout()

## 4 · Autodiff I — the dispersion, for free

Because the trace is a differentiable JAX function, the local dispersion **dy/dλ** (pixels per micron) is just `jax.grad` of it — no finite differences, no tabulation. This is exactly the tool notebook 04 used to convert a boxcar sum into a calibrated spectrum.

In [ ]:
def trace_y(wl_scalar):
    wl1 = jnp.atleast_1d(wl_scalar)
    xfa, yfa = omj.sca_to_fpa(payload, jnp.array([2000.0]), jnp.array([2000.0]))
    xm, ym = omj.trace_beam(payload, jnp.broadcast_to(xfa, wl1.shape),
                            jnp.broadcast_to(yfa, wl1.shape), wl1)
    _, t = omj.mpa_to_sca(payload, xm, ym)
    return t[0]

dy_dlam = jax.jit(jax.vmap(jax.grad(trace_y)))(wl)
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(np.asarray(wl), np.abs(np.asarray(dy_dlam)), color="C2")
ax.set(xlabel="wavelength [µm]", ylabel="|dy/dλ| [pix/µm]",
       title="local dispersion, by autodiff")
fig.tight_layout()
print(f"≈ {np.abs(np.asarray(dy_dlam)).mean():.0f} pix/µm  ⇒  {1e4/np.abs(np.asarray(dy_dlam)).mean():.1f} Å/pix")

## 5 · Autodiff II — the dispersion Jacobian

Now differentiate with respect to *position*. Hold the wavelength fixed and ask how the dispersed image point $(x', y')$ moves when the source moves: the 2×2 **Jacobian** $J = \partial(x',y')/\partial(x,y)$. It is close to the identity with a small shear — and it is precisely the local linear map `galaxy_disperser` applies to warp a galaxy stamp at each wavelength (notebook 05). `jax.jacobian` gives it directly.

In [ ]:
def disperse_map(xsca, ysca, wl_scalar):
    xfa, yfa = omj.sca_to_fpa(payload, jnp.atleast_1d(xsca), jnp.atleast_1d(ysca))
    xm, ym = omj.trace_beam(payload, xfa, yfa, jnp.atleast_1d(wl_scalar))
    xo, yo = omj.mpa_to_sca(payload, xm, ym)
    return jnp.array([xo[0], yo[0]])

jac = jax.jit(jax.jacobian(disperse_map, argnums=(0, 1)))
Jx, Jy = jac(2000.0, 2000.0, 1.5)        # columns: d/dx and d/dy
J = np.array([[float(Jx[0]), float(Jy[0])], [float(Jx[1]), float(Jy[1])]])
print("Jacobian J = ∂(x',y')/∂(x,y) at (2000,2000), 1.5 µm:")
print(np.round(J, 4))
print("\nJ − I (the shear/scale the galaxy disperser applies):")
print(np.round(J - np.eye(2), 4))

The diagonal is ≈ 1 (the image is barely magnified) and the off-diagonal shear is ≈ 1 %. A finite-difference check matches the **diagonal**, but the tiny off-diagonal terms sit near the float32 noise floor for differencing — so the autodiff value is not just more convenient, it is *more accurate*. (This is the practical reason the optical model is built in JAX.)

In [ ]:
# finite-difference comparison (note the off-diagonal is where differencing struggles)
h = 0.05
f0 = np.asarray(disperse_map(2000.0, 2000.0, 1.5))
fd_dx = (np.asarray(disperse_map(2000.0 + h, 2000.0, 1.5)) - f0) / h
fd_dy = (np.asarray(disperse_map(2000.0, 2000.0 + h, 1.5)) - f0) / h
J_fd = np.array([[fd_dx[0], fd_dy[0]], [fd_dx[1], fd_dy[1]]])
print("autodiff J:\n", np.round(J, 4))
print("finite-diff J:\n", np.round(J_fd, 4))

In [ ]:
# how much does the warp vary across the detector? map ||J - I|| at fixed wavelength.
grid = np.linspace(200, 3900, 12)
jac_v = jax.jit(jax.vmap(jax.vmap(
    lambda xx, yy: jax.jacobian(disperse_map, argnums=(0, 1))(xx, yy, 1.5),
    in_axes=(0, None)), in_axes=(None, 0)))
GX, GY = jnp.array(grid), jnp.array(grid)
Jxg, Jyg = jac_v(GX, GY)                  # each [ny, nx, 2]
Jgrid = np.stack([np.asarray(Jxg), np.asarray(Jyg)], axis=-1)   # [ny,nx,2(out),2(in)]
frob = np.sqrt(((Jgrid - np.eye(2)) ** 2).sum(axis=(-1, -2)))

fig, ax = plt.subplots(figsize=(5.2, 4.4))
im = ax.imshow(frob, origin="lower", extent=[grid[0], grid[-1], grid[0], grid[-1]], cmap="viridis")
fig.colorbar(im, ax=ax, label="‖J − I‖")
ax.set(xlabel="x [pix]", ylabel="y [pix]", title="dispersion warp across SCA5 (1.5 µm)")
fig.tight_layout()

## 6 · jit and vmap — the same code at scale

The transforms are ordinary JAX, so `jax.jit` compiles them and `jax.vmap` vectorizes them over arbitrarily many inputs — no Python loop. This is why notebook 06 could disperse a whole catalog, and why the identical code runs on a GPU.

In [ ]:
import time
n = 100_000
xs = jnp.asarray(np.random.default_rng(0).uniform(1, 4088, n))
ys = jnp.asarray(np.random.default_rng(1).uniform(1, 4088, n))

sky = jax.jit(lambda a, b: omj.sca_to_fpa(payload, a, b))
sky(xs[:1], ys[:1])[0].block_until_ready()                 # warm up
t = time.time(); r = sky(xs, ys); jax.block_until_ready(r)
print(f"{n:,} sca→fpa transforms in {1e3*(time.time()-t):.1f} ms on {jax.default_backend()}")

## Recap

- The optical model is three coordinate frames (**SCA px ↔ FPA deg ↔ MPA mm**) plus a wavelength-dependent **trace**, all as differentiable JAX functions.
- `jax.grad` gives the **dispersion** (notebook 04); `jax.jacobian` gives the **warp** that shapes extended sources (notebook 05); `jax.jit`/`jax.vmap` make it run at **catalog scale** and on GPUs (notebook 06).
- Autodiff isn't just convenient here — for the small Jacobian shear it is more accurate than finite differences.

That small, differentiable, vectorized core is what every disperser and the extractor are built on.

**Next — [08 · Scaling out on a GPU](08_gpu_scale_out.ipynb).** The whole field, all three orders, at GPU scale.